# Task 3b: Held-Out TDK Evaluation

This notebook evaluates the single checkpoint selected using **non-TDK validation WER** in Task 3a. It does not tune parameters on TDK. The baseline `generated_text` is preserved, and `generated_text_ft` is added. Python helpers implement resumable inference; this notebook records the experiment and comparison.

Status: execution and final results pending. Do not treat the scaffold or a smoke run as completed evaluation.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'asr-train':
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / 'asr-train'), str(ROOT / 'asr')]
BASE = Path(os.environ.get('TDK_BASE', ROOT / 'asr/TDK_subset.csv'))
MODEL = Path(os.environ.get('YCSEP_MODEL', ROOT / 'asr-train/parakeet-tdt-0.6b-v3-ycsep.nemo'))
OUTPUT = Path(os.environ.get('TDK_EVALUATION', ROOT / 'test_docs/test/runtime/final-tdk-evaluation'))
CACHE = os.environ.get('TDK_CACHE') or None


## Audio and Reproducibility
Prefer the verified MP3 caches from Task 2c so the two models see identical audio. If unavailable, the helper extracts the original YCSEP WAV timestamp range and encodes MP3 using the same fallback as Task 2c. This is not byte-identical to the CSV's original MP3; report which path was used. Both paths use librosa mono 16 kHz loading followed by PCM WAV, matching the API preprocessing. The checkpoint and baseline CSV hashes bind the resume journal.

Before the final run, confirm that the baseline has **all 218,011 TDK rows** and zero API errors. Empty successful transcriptions are valid model outputs, not missing rows.

In [ ]:
import csv
with BASE.open(encoding='utf-8-sig', newline='') as handle:
    baseline_rows = list(csv.DictReader(handle))
assert len(baseline_rows) == 218011, 'Final TDK baseline is incomplete'
assert all(r['channel'] == 'The_Daily_Ketchup_Podcast' and not r['asr_error'] for r in baseline_rows)
assert MODEL.is_file(), 'Selected Task 3a model is required'
print({'baseline_rows': len(baseline_rows), 'checkpoint': MODEL.name})


In [ ]:
from evaluate_tdk import evaluate
result = evaluate(BASE, MODEL, OUTPUT, batch_size=8, workers=8, cache=CACHE)
result


## Paired Metrics
WER is the primary metric, calculated as pooled substitutions + deletions + insertions divided by total reference words, not the mean of row WER. CER is secondary. The same Unicode NFKC/case-fold/word normalization is applied to both systems; CER excludes spaces. Empty-reference rows retain insertion counts in the pooled numerator. Failed inference must be resolved before declaring final performance.

In [ ]:
from score_transcriptions import score
raw = OUTPUT / 'TDK_subset_ft.csv'
base_metrics = score(raw, OUTPUT / 'TDK_base_scored.csv')
ft_metrics = score(OUTPUT / 'TDK_base_scored.csv', OUTPUT / 'TDK_subset.csv',
                   prediction='generated_text_ft', suffix='ft', error_column='asr_error_ft')
assert base_metrics['coverage'] == ft_metrics['coverage'] == 1
comparison = {'base_wer': base_metrics['corpus_wer'], 'ft_wer': ft_metrics['corpus_wer'],
              'base_cer': base_metrics['corpus_cer'], 'ft_cer': ft_metrics['corpus_cer']}
comparison['absolute_wer_change'] = comparison['ft_wer'] - comparison['base_wer']
comparison['relative_wer_reduction'] = (1-comparison['ft_wer']/comparison['base_wer']) if comparison['base_wer'] else None
(OUTPUT / 'comparison.json').write_text(json.dumps(comparison, indent=2), encoding='utf-8')
comparison


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, metric in zip(axes, ('wer', 'cer')):
    ax.bar(['Base', 'Fine-tuned'], [comparison['base_'+metric], comparison['ft_'+metric]], color=['#477c91', '#c4654e'])
    ax.set_ylabel(metric.upper())
    ax.set_ylim(bottom=0)
fig.tight_layout()
plt.show()


## Interpretation to Complete After Execution
Record actual WER/CER, whether fine-tuning improves or regresses, and inspect representative improvements and regressions (Singlish, names, code-switching, short/overlapping speech). Treat repeated clips from a video as correlated observations when estimating uncertainty. Task 4 reports the comparison and limitations; it is not a second checkpoint-selection stage. No improvement is assumed in advance.